# Dataset Analysis

In [74]:
import pprint
import pydytuesday
import pandas
from IPython.display import display
import json
import altair as alt

country_lists = pandas.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-09-09/country_lists.csv')
rank_by_year = pandas.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2025/2025-09-09/rank_by_year.csv')

print("Country Lists Dataset")
display(country_lists.head())

print("Rank by Year Dataset")
display(rank_by_year.head())

andorra_visa_required = country_lists.loc[country_lists['country'] == 'Andorra', 'visa_required'].values[0]
andorra_visa_required_json = json.loads(andorra_visa_required)
print("Andorra citizens require visa to enter:")
# Print only the first 3 countries for brevity
pprint.pprint(andorra_visa_required_json[0][:3])
andorra_visa_free_access = country_lists.loc[country_lists['country'] == 'Andorra', 'visa_free_access'].values[0]
andorra_visa_free_access_json = json.loads(andorra_visa_free_access)
visa_free_access_count = len(andorra_visa_free_access_json[0])
print(f"Number of countries that andorran passport can enter without visa: {visa_free_access_count}. There are other countries that provide visa on arrival or electronic.")

print(f"Total number of countries in the dataset: {country_lists.shape[0]}")

Country Lists Dataset


,code,country,visa_required,visa_online,visa_on_arrival,visa_free_access,electronic_travel_authorisation
0,PS,Palestinian Territory,"[[{""code"":""AF"",""name"":""Afghanistan""},{""code"":""...","[[{""code"":""AG"",""name"":""Antigua and Barbuda""},{...","[[{""code"":""BD"",""name"":""Bangladesh""},{""code"":""B...","[[{""code"":""BO"",""name"":""Bolivia""},{""code"":""CK"",...","[[{""code"":""LK"",""name"":""Sri Lanka""},{""code"":""KE..."
1,AD,Andorra,"[[{""code"":""AF"",""name"":""Afghanistan""},{""code"":""...","[[{""code"":""AO"",""name"":""Angola""},{""code"":""AZ"",""...","[[{""code"":""BH"",""name"":""Bahrain""},{""code"":""BD"",...","[[{""code"":""JP"",""name"":""Japan""},{""code"":""AL"",""n...","[[{""code"":""AU"",""name"":""Australia""},{""code"":""CA..."
2,VA,Vatican City,"[[{""code"":""AF"",""name"":""Afghanistan""},{""code"":""...","[[{""code"":""AZ"",""name"":""Azerbaijan""},{""code"":""B...","[[{""code"":""BH"",""name"":""Bahrain""},{""code"":""BD"",...","[[{""code"":""AL"",""name"":""Albania""},{""code"":""AD"",...","[[{""code"":""AU"",""name"":""Australia""},{""code"":""CA..."
3,SM,San Marino,"[[{""code"":""AF"",""name"":""Afghanistan""},{""code"":""...","[[{""code"":""AZ"",""name"":""Azerbaijan""},{""code"":""B...","[[{""code"":""BH"",""name"":""Bahrain""},{""code"":""BD"",...","[[{""code"":""JP"",""name"":""Japan""},{""code"":""AL"",""n...","[[{""code"":""AU"",""name"":""Australia""},{""code"":""CA..."
4,MC,Monaco,"[[{""code"":""AF"",""name"":""Afghanistan""},{""code"":""...","[[{""code"":""AZ"",""name"":""Azerbaijan""},{""code"":""B...","[[{""code"":""BH"",""name"":""Bahrain""},{""code"":""BD"",...","[[{""code"":""JP"",""name"":""Japan""},{""code"":""AL"",""n...","[[{""code"":""AU"",""name"":""Australia""},{""code"":""CA..."


Rank by Year Dataset


,code,country,region,rank,visa_free_count,year
0,AF,Afghanistan,ASIA,116,26,2021
1,AF,Afghanistan,ASIA,106,26,2020
2,AF,Afghanistan,ASIA,106,30,2018
3,AF,Afghanistan,ASIA,104,24,2017
4,AF,Afghanistan,ASIA,104,25,2016


Andorra citizens require visa to enter:
[{'code': 'AF', 'name': 'Afghanistan'},
 {'code': 'DZ', 'name': 'Algeria'},
 {'code': 'BT', 'name': 'Bhutan'}]
Number of countries that andorran passport can enter without visa: 120. There are other countries that provide visa on arrival or electronic.
Total number of countries in the dataset: 199


## Dataset Cleaning

### Initial dataset merge

In [75]:
# rank by year has a single column called 'year' with years from 2006 to 2024 or so.
# I will add year_X to the country_lists dataset for each year X from min(unique_year) to max(unique_year)

unique_years = rank_by_year['year'].unique()
for year in unique_years:
    year_column_name = f'year_visa_free_count_{year}'
    country_lists[year_column_name] = country_lists['country'].map(
        rank_by_year.loc[rank_by_year['year'] == year].set_index('country')['visa_free_count']
    )


# Add also the region (continent) to the country_lists dataframe
country_lists = country_lists.merge(
    rank_by_year[['country', 'region']].drop_duplicates(),
    on='country',
    how='left'
)

### Add ISO 3166-1 codes to each country

In [76]:
import pycountry

# Create a function to get ISO 3166-1 numeric code from country name
def get_iso_numeric_code(country_name):
    try:
        # Try direct lookup
        country = pycountry.countries.get(name=country_name)
        if country:
            return int(country.numeric)
        
        # Try fuzzy search
        country = pycountry.countries.search_fuzzy(country_name)[0]
        return int(country.numeric)
    except:
        return None

# Add ISO 3166-1 numeric codes to rank_by_year
rank_by_year['iso_numeric'] = rank_by_year['country'].apply(get_iso_numeric_code).astype('Int64')

# Add ISO 3166-1 numeric codes to country_lists
country_lists['iso_numeric'] = country_lists['country'].apply(get_iso_numeric_code).astype('Int64')

# Check results
print(f"Countries with ISO codes: {rank_by_year['iso_numeric'].notna().sum()}")
print(f"Countries without ISO codes: {rank_by_year['iso_numeric'].isna().sum()}")

# Display countries without codes for manual mapping if needed
missing_codes = rank_by_year[rank_by_year['iso_numeric'].isna()]['country'].unique()
if len(missing_codes) > 0:
    print("\nCountries without ISO codes:")
    print(missing_codes)

display(rank_by_year[['country', 'iso_numeric']].drop_duplicates())


Countries with ISO codes: 3720
Countries without ISO codes: 230

Countries without ISO codes:
['Cape Verde Islands' 'Comoro Islands' 'Congo (Rep.)' 'Congo (Dem. Rep.)'
 'Hong Kong (SAR China)' 'Macao (SAR China)' 'Palau Islands'
 'St. Kitts and Nevis' 'St. Lucia' 'St. Vincent and the Grenadines'
 'Taiwan (Chinese Taipei)' 'Palestinian Territory']


,country,iso_numeric
0,Afghanistan,4
20,Albania,8
40,Algeria,12
60,Angola,24
80,Antigua and Barbuda,28
...,...,...
3854,Monaco,492
3874,San Marino,674
3894,Vatican City,336
3912,Andorra,20


### Add normalized population per country

We will use the `pycountry` library to extract population data for each country. However, note that `pycountry` doesn't directly provide population data. We'll need to use an external dataset or API for population information. For this analysis, we'll use the World Bank population data or a similar source to add normalized population factors to rank_by_year dataset.

In [77]:
import requests

# Since pycountry doesn't provide population data, we'll use World Bank data
# We can use the wbgapi library or fetch World Bank data directly


# Fetch population data from World Bank API for latest available year
wb_url = "https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?format=json&per_page=500&date=2021"

try:
    response = requests.get(wb_url)
    wb_data = response.json()
    
    # Extract population data
    population_data = []
    if len(wb_data) > 1:
        for entry in wb_data[1]:
            if entry['value'] is not None:
                population_data.append({
                    'country_name': entry['country']['value'],
                    'iso3_code': entry['countryiso3code'],
                    'population': entry['value'],
                    'year': entry['date']
                })
    
    pop_df = pandas.DataFrame(population_data)
    
    # Create a mapping function to match country names
    def match_population(country_name, iso_numeric):
        # Try to match by ISO code first
        if pandas.notna(iso_numeric):
            country = pycountry.countries.get(numeric=str(int(iso_numeric)).zfill(3))
            if country and hasattr(country, 'alpha_3'):
                match = pop_df[pop_df['iso3_code'] == country.alpha_3]
                if not match.empty:
                    return match.iloc[0]['population']
        
        # Try direct name match
        match = pop_df[pop_df['country_name'] == country_name]
        if not match.empty:
            return match.iloc[0]['population']
        
        return None
    
    # Add population to rank_by_year
    rank_by_year['population'] = rank_by_year.apply(
        lambda row: match_population(row['country'], row['iso_numeric']), 
        axis=1
    )
    # I will also add the population to the country_lists dataframe
    country_lists = country_lists.merge(
        rank_by_year[['country', 'population']].drop_duplicates(),
        on='country',
        how='left'
    )
    
    # Calculate normalized metrics
    rank_by_year['visa_free_per_million'] = (rank_by_year['visa_free_count'] / 
                                               (rank_by_year['population'] / 1_000_000)).round(2)
    
    rank_by_year['visa_free_per_population'] = (rank_by_year['visa_free_count'] *
                                               rank_by_year['population']).round(6)

    # Store normalized visa-free counts based on population
        # Get biggest rank_by_year['visa_free_per_million']
    max_visa_free_per_million = rank_by_year['visa_free_per_million'].max()
    rank_by_year['visa_free_per_million_normalized'] = ( rank_by_year['visa_free_per_million'] / max_visa_free_per_million * 100 ).round(2) 


    # This is a bit triky, but now i can add the visa_free_per_million and visa_free_per_population to country_lists
    unique_years = rank_by_year['year'].unique()
    for year in unique_years:
        # Add visa_free_count column
        count_col = f'year_visa_free_per_million_{year}'
        country_lists[count_col] = country_lists['country'].map(
            rank_by_year.loc[rank_by_year['year'] == year].set_index('country')['visa_free_per_million']
        )
        # Add per_million column
        per_million_col = f'year_visa_free_per_population_{year}'
        country_lists[per_million_col] = country_lists['country'].map(
            rank_by_year.loc[rank_by_year['year'] == year].set_index('country')['visa_free_per_population']
        )

    print(f"Countries with population data: {rank_by_year['population'].notna().sum()}")
    print(f"Countries without population data: {rank_by_year['population'].isna().sum()}")
    
    # Display sample with population data
    display(rank_by_year[['country', 'year', 'visa_free_count', 'population', 'visa_free_per_million']].head(10))
    
except Exception as e:
    print(f"Error fetching World Bank data: {e}")
    print("Adding placeholder population column")
    rank_by_year['population'] = None

Countries with population data: 3762
Countries without population data: 188


,country,year,visa_free_count,population,visa_free_per_million
0,Afghanistan,2021,26,40000412.0,0.65
1,Afghanistan,2020,26,40000412.0,0.65
2,Afghanistan,2018,30,40000412.0,0.75
3,Afghanistan,2017,24,40000412.0,0.60
4,Afghanistan,2016,25,40000412.0,0.62
5,Afghanistan,2015,25,40000412.0,0.62
6,Afghanistan,2014,28,40000412.0,0.70
7,Afghanistan,2013,28,40000412.0,0.70
8,Afghanistan,2012,26,40000412.0,0.65
9,Afghanistan,2011,24,40000412.0,0.60


### Add 2006 and 2025 increase


In [78]:
# Calculate the change between 2006 and 2021 for each country
data_2006 = rank_by_year[rank_by_year['year'] == 2006][['country', 'visa_free_count']].rename(columns={'visa_free_count': 'visa_free_2006'})
data_2021 = rank_by_year[rank_by_year['year'] == 2021][['country', 'visa_free_count']].rename(columns={'visa_free_count': 'visa_free_2021'})

# Merge to calculate change
change_df = data_2006.merge(data_2021, on='country', how='outer')
change_df['visa_free_change_2006_2021'] = (change_df['visa_free_2021'] - change_df['visa_free_2006']).abs()

# Merge back to rank_by_year
rank_by_year = rank_by_year.merge(change_df[['country', 'visa_free_change_2006_2021']], on='country', how='left')

print(f"Added visa_free_change_2006_2021 column")
display(rank_by_year[['country', 'year', 'visa_free_count', 'visa_free_change_2006_2021']].head(20))


# For each year I want to also store how much it changed from previous year
rank_by_year = rank_by_year.sort_values(by=['country', 'year'])
rank_by_year['visa_free_change_from_previous_year'] = rank_by_year.groupby('country')['visa_free_count'].diff().fillna(0).abs()
print(f"Added visa_free_change_from_previous_year column")
display(rank_by_year[['country', 'year', 'visa_free_count', 'visa_free_change_from_previous_year']].head(20))


Added visa_free_change_2006_2021 column


,country,year,visa_free_count,visa_free_change_2006_2021
0,Afghanistan,2021,26,14.0
1,Afghanistan,2020,26,14.0
2,Afghanistan,2018,30,14.0
3,Afghanistan,2017,24,14.0
4,Afghanistan,2016,25,14.0
5,Afghanistan,2015,25,14.0
6,Afghanistan,2014,28,14.0
7,Afghanistan,2013,28,14.0
8,Afghanistan,2012,26,14.0
9,Afghanistan,2011,24,14.0


Added visa_free_change_from_previous_year column


,country,year,visa_free_count,visa_free_change_from_previous_year
14,Afghanistan,2006,12,0.0
13,Afghanistan,2007,0,12.0
12,Afghanistan,2008,22,22.0
11,Afghanistan,2009,0,22.0
10,Afghanistan,2010,26,26.0
9,Afghanistan,2011,24,2.0
8,Afghanistan,2012,26,2.0
7,Afghanistan,2013,28,2.0
6,Afghanistan,2014,28,0.0
5,Afghanistan,2015,25,3.0


### Number of Arrivals
We now will calculate how many countries we can go with P passport.

I will use this webpage to check if i am correct https://embassies.net/andorra-visa-exemption

In [79]:
# country_lists has country column. it also has the following "visa free" related columns:
#   visa_online, 
#   visa_on_arrival, 
#   visa_free_access
#   electronic_travel_authorisation 

# We iterate over each country c
    # we iterate over the countries!=c
    # we check if c is in any column considered "visa free" for that country
    # if yes, we increment a counter


# unique countries in country_lists
C = country_lists['country'].unique()


def dictionary_to_set(d):
    """
    we are actually provided a dictionary with the following format:
    [{'code': 'JP', 'name': 'Japan'}, {'code': 'LU', 'name': 'Luxembourg'}, ...]
    
    return a set with only the country names
    """
    result = set()
    for entry in d:
        result.add(entry['name'])
    return result

for c in C:
    count = 0
    for index, row in country_lists.iterrows():
        if row['country'] == c:
            continue
        visa_free_access = dictionary_to_set(json.loads(row['visa_free_access'])[0])
        visa_on_arrival = dictionary_to_set(json.loads(row['visa_on_arrival'])[0])
        visa_online = dictionary_to_set(json.loads(row['visa_online'])[0])
        electronic_travel_authorisation = dictionary_to_set(json.loads(row['electronic_travel_authorisation'])[0])
        visa_required = dictionary_to_set(json.loads(row['visa_required'])[0])
        
        # create a set of all visa free countries for that country
        all_visa_free = visa_free_access | visa_on_arrival | visa_online | electronic_travel_authorisation
        
        if c in all_visa_free:
            count += 1
    country_lists.loc[country_lists['country'] == c, 'visa_free_arrivals_count'] = count
    
# I will add this new column to rank_by_year
rank_by_year = rank_by_year.merge(country_lists[['country', 'visa_free_arrivals_count']], on='country', how='left')
print(f"Added visa_free_arrivals_count column")

Added visa_free_arrivals_count column


In [80]:
# Iterate over country_lists and discover which type of access they have to enter the US
us_access_list = []

for index, row in country_lists.iterrows():
    country = row['country']
    
    visa_free_access = dictionary_to_set(json.loads(row['visa_free_access'])[0])
    visa_on_arrival = dictionary_to_set(json.loads(row['visa_on_arrival'])[0])
    visa_online = dictionary_to_set(json.loads(row['visa_online'])[0])
    electronic_travel_authorisation = dictionary_to_set(json.loads(row['electronic_travel_authorisation'])[0])
    visa_required = dictionary_to_set(json.loads(row['visa_required'])[0])
    
    # Determine US access type
    us_access_type = None
    if "United States" in visa_free_access:
        us_access_type = "visa_free_access"
    elif "United States" in visa_on_arrival:
        us_access_type = "visa_on_arrival"
    elif "United States" in visa_online:
        us_access_type = "visa_online"
    elif "United States" in electronic_travel_authorisation:
        us_access_type = "electronic_travel_authorisation"
    elif "United States" in visa_required:
        us_access_type = "visa_required"
    elif country == "United States":
        us_access_type = "US"
    else:
        us_access_type = "unknown"
        
        
        
    us_access_list.append({
        'country': country,
        'us_access_type': us_access_type
    })

# Create a DataFrame with US access types
us_access_df = pandas.DataFrame(us_access_list)

print(f"Total countries: {len(us_access_df)}")
print("\nUS Access Type Distribution:")
print(us_access_df['us_access_type'].value_counts())

display(us_access_df.head(20))

# i will now merge this us_access_df with country_lists
country_lists = country_lists.merge(us_access_df, on='country', how='left')
print(f"Merged us_access_type into country_lists")


# for comodity i will translate the string to an integer code on what i belive is restriction level
# 0 = visa_required
# 1 = electronic_travel_authorisation
# 2 = visa_online
# 3 = visa_on_arrival
# 4 = visa_free_access

us_access_type_map = {
    'visa_required': 0,
    'electronic_travel_authorisation': 1,
    'visa_online': 2,
    'visa_on_arrival': 3,
    'visa_free_access': 4,
    'unknown': 5,
    'US': 6
}

# Clean up duplicate us_access_type columns if they exist
if 'us_access_type_x' in country_lists.columns:
    country_lists['us_access_type'] = country_lists['us_access_type_x']
    country_lists = country_lists.drop(columns=['us_access_type_x', 'us_access_type_y'])

country_lists['us_access_type_code'] = country_lists['us_access_type'].map(us_access_type_map)
print(f"Added us_access_type_code column")
display(country_lists[['country', 'us_access_type', 'us_access_type_code']].head(20))


Total countries: 199

US Access Type Distribution:
us_access_type
visa_required                      152
electronic_travel_authorisation     42
visa_free_access                     4
US                                   1
Name: count, dtype: int64


,country,us_access_type
0,Palestinian Territory,visa_required
1,Andorra,electronic_travel_authorisation
2,Vatican City,visa_required
3,San Marino,electronic_travel_authorisation
4,Monaco,electronic_travel_authorisation
5,Liechtenstein,electronic_travel_authorisation
6,Zimbabwe,visa_required
7,Zambia,visa_required
8,Yemen,visa_required
9,Vietnam,visa_required


Merged us_access_type into country_lists
Added us_access_type_code column


,country,us_access_type,us_access_type_code
0,Palestinian Territory,visa_required,0
1,Andorra,electronic_travel_authorisation,1
2,Vatican City,visa_required,0
3,San Marino,electronic_travel_authorisation,1
4,Monaco,electronic_travel_authorisation,1
5,Liechtenstein,electronic_travel_authorisation,1
6,Zimbabwe,visa_required,0
7,Zambia,visa_required,0
8,Yemen,visa_required,0
9,Vietnam,visa_required,0


### Add Population

In [81]:
import requests

# Since pycountry doesn't provide population data, we'll use World Bank data
# We can use the wbgapi library or fetch World Bank data directly


# Fetch population data from World Bank API for latest available year
wb_url = "https://api.worldbank.org/v2/country/all/indicator/SP.POP.TOTL?format=json&per_page=500&date=2021"

try:
    response = requests.get(wb_url)
    wb_data = response.json()
    
    # Extract population data
    population_data = []
    if len(wb_data) > 1:
        for entry in wb_data[1]:
            if entry['value'] is not None:
                population_data.append({
                    'country_name': entry['country']['value'],
                    'iso3_code': entry['countryiso3code'],
                    'population': entry['value'],
                    'year': entry['date']
                })
    
    pop_df = pandas.DataFrame(population_data)
    
    # Create a mapping function to match country names
    def match_population(country_name, iso_numeric):
        # Try to match by ISO code first
        if pandas.notna(iso_numeric):
            country = pycountry.countries.get(numeric=str(int(iso_numeric)).zfill(3))
            if country and hasattr(country, 'alpha_3'):
                match = pop_df[pop_df['iso3_code'] == country.alpha_3]
                if not match.empty:
                    return match.iloc[0]['population']
        
        # Try direct name match
        match = pop_df[pop_df['country_name'] == country_name]
        if not match.empty:
            return match.iloc[0]['population']
        
        return None
    
    # Add population to rank_by_year
    rank_by_year['population'] = rank_by_year.apply(
        lambda row: match_population(row['country'], row['iso_numeric']), 
        axis=1
    )
    
    # Calculate normalized metrics
    rank_by_year['visa_free_per_million'] = (rank_by_year['visa_free_count'] / 
                                               (rank_by_year['population'] / 1_000_000)).round(2)
    
    print(f"Countries with population data: {rank_by_year['population'].notna().sum()}")
    print(f"Countries without population data: {rank_by_year['population'].isna().sum()}")
    
    # Display sample with population data
    display(rank_by_year[['country', 'year', 'visa_free_count', 'population', 'visa_free_per_million']].head(10))
    
except Exception as e:
    print(f"Error fetching World Bank data: {e}")
    print("Adding placeholder population column")
    rank_by_year['population'] = None

Countries with population data: 3762
Countries without population data: 188


,country,year,visa_free_count,population,visa_free_per_million
0,Afghanistan,2006,12,40000412.0,0.30
1,Afghanistan,2007,0,40000412.0,0.00
2,Afghanistan,2008,22,40000412.0,0.55
3,Afghanistan,2009,0,40000412.0,0.00
4,Afghanistan,2010,26,40000412.0,0.65
5,Afghanistan,2011,24,40000412.0,0.60
6,Afghanistan,2012,26,40000412.0,0.65
7,Afghanistan,2013,28,40000412.0,0.70
8,Afghanistan,2014,28,40000412.0,0.70
9,Afghanistan,2015,25,40000412.0,0.62


## Export dataset for later use

In [110]:
# export the datasets for later use
country_lists.to_csv('country_lists_enriched.csv', index=False)


# Q1: 1. Which continent has the most visa-free destinations? How does this vary when considering population size?

## Understanding the question:
In the dataset **Henley Passport Index** we get entries such as:
- country = Afganistan
- Visa_free_count = 26

This value represent how many countries can Afganistan go to (destinaiton) without a previous visa. We can see this in the official page https://www.henleyglobal.com/passport-index/improve




## My Approach

A continent is not really representative of how mobility works per country so I decided to represent the distribution of each continent to provide more information.

Furthermore, to answer which continent has the most visa free, I calculate the mean and represent it as a vertical line inside the distributions.

Because each country has different population, we can multiply the population by the visa free destinations to know how much people can actually move freely to other countries. As an example, even if china has only 83 destinations, it has 1,412,360,000 people that can use it; The weighted score will be greater than smaller countries with higher mobility. Additionally we can show how accessible the world is for the average citizen by using the visa_free_per_million.


I added a dropdown to switch between raw counts, population-weighted counts, and per-million metrics, helping users compare absolute and normalized mobility. A year slider enables temporal exploration. To improve legibility, I used a categorized color palette for continents. The y-axis uses a power scale to highlight differences among countries with lower values. This design reduces clutter compared to bar charts and allows users to answer the question by observing both the spread and average of mobility within continents. Alternatives like stacked bars or simple averages hid important distribution details and outliers, so I opted for a strip plot with interactive controls for clarity and flexibility. Other contemplated charts where histograms (observe distribution per continent).

In [84]:
# prepare dataframe
df = country_lists.copy()
# Melt each metric separately
visa_free_count = df.melt(
    id_vars=['country', 'region'],
    value_vars=[col for col in df.columns if col.startswith('year_visa_free_count_')],
    var_name='year',
    value_name='visa_free_count'
)
visa_free_count['year'] = visa_free_count['year'].str.replace('year_visa_free_count_', '')

per_million = df.melt(
    id_vars=['country', 'region'],
    value_vars=[col for col in df.columns if col.startswith('year_visa_free_per_million_')],
    var_name='year',
    value_name='visa_free_per_million'
)
per_million['year'] = per_million['year'].str.replace('year_visa_free_per_million_', '')

normalized = df.melt(
    id_vars=['country', 'region'],
    value_vars=[col for col in df.columns if col.startswith('year_visa_free_per_population_')],
    var_name='year',
    value_name='visa_free_per_population'
)
normalized['year'] = normalized['year'].str.replace('year_visa_free_per_population_', '')

# Merge all three melted DataFrames on country, region, and year
long_df = visa_free_count.merge(per_million, on=['country', 'region', 'year'])
long_df = long_df.merge(normalized, on=['country', 'region', 'year'])

# change year from string to integer
long_df['year'] = long_df['year'].astype(int) 

In [98]:
# Create a dropdown selection for the metric
metric_dropdown = alt.binding_select(
    options=['visa_free_count', 'visa_free_per_population', 'visa_free_per_million'],
    labels=['Visa-Free Count', 'Visa-Free × Population', 'Visa-Free per Million'],
    name='Metric: '
)
metric_select = alt.selection_point(
    name='metric_selection',
    fields=['metric'],
    bind=metric_dropdown,
    value='visa_free_count'
)


# Define a reusable color object for continent/region
continent_color = alt.Color(
    'region:N',
    title='Continent',
)

# Interactive strip plot with year slider and metric dropdown
year_slider = alt.binding_range(min=2006, max=2025, step=1, name='Year: ')
year_select = alt.selection_point(name='year_selection', fields=['year'], bind=year_slider, value=2021)

strip_chart = alt.Chart(long_df).transform_fold(
    ['visa_free_count', 'visa_free_per_population', 'visa_free_per_million'],
    as_=['metric', 'value']
).mark_circle(size=60, opacity=0.6).encode(
    x=alt.X('region:N', title='Continent'),
    y=alt.Y(
        'value:Q',
        title='Visa-Free Destinations',
        scale=alt.Scale(type='pow', exponent=0.5),
        axis=alt.Axis(labelLimit=200, labelFontSize=10, labelPadding=10, format="~s")  # add these options
    ),
    color=continent_color,
    tooltip=['country', 'region', 'year', 'visa_free_count', 'visa_free_per_population', 'visa_free_per_million'],
    xOffset=alt.XOffset('jitter:Q', scale=alt.Scale(domain=[-0.2, 0.2]))
).add_params(
    year_select,
    metric_select
).transform_filter(
    year_select
).transform_filter(
    metric_select
).transform_calculate(
    jitter="((indexof('ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz', substring(datum.country, 0, 1)) % 20) / 100 - 0.1)"
).properties(
    title='Visa-Free Destinations by Continent (Select Year and Metric)',
    width=300,
    height=400
)

mean_line = alt.Chart(long_df).transform_fold(
    ['visa_free_count', 'visa_free_per_population', 'visa_free_per_million'],
    as_=['metric', 'value']
).mark_rule(
    color='black',
    strokeDash=[4,2]
).encode(
    x=alt.X('region:N'),
    y='mean(value):Q',
    tooltip=[alt.Tooltip('region:N'), alt.Tooltip('mean(value):Q', title='Mean')],
).add_params(
    year_select,
    metric_select
).transform_filter(
    year_select
).transform_filter(
    metric_select
)

# Combine strip plot and mean lines
q1 = (strip_chart + mean_line)
q1

alt.LayerChart(...)

# Q2: Which countries have experienced the greatest changes as visa-free countries between 2006 and 2021?

## Understanding the request
The goal was to identify which countries experienced the greatest changes in visa-free access between 2006 and 2021. Initially, I misunderstood the question, thinking it referred to where citizens of a country could travel visa-free. In fact, it refers to how many countries can now enter a specific country without a visa. Because the available dataset (country_lists.csv) does not include historical visa data, I limited the analysis to visa-free "exits," not "entries."

## My Approach
Given 199 countries across 15 years, plotting all data would create excessive visual noise. To address this, I focused on displaying only a selected range of countries—typically the top few with the greatest changes—using interactive sliders to adjust which ranks are shown. The visualization combines two complementary plots: a horizontal bar chart and a slope chart. The bar chart highlights the change in visa-free destinations from 2006 to 2021, while the slope chart preserves information about the start and end values. Colors represent continents to retain regional context.

Although this method loses details about changes within intermediate years, it provides a clear and interactive overview of how overall visa-free access evolved between 2006 and 2021, emphasizing both absolute levels and differences across regions.

In [ ]:
data = country_lists.copy()

data_selected = data[['country', 'year_visa_free_count_2006', 'year_visa_free_count_2021', 'region']].copy()
data_selected = data_selected.rename(columns={
    'year_visa_free_count_2006': 'count_2006',
    'year_visa_free_count_2021': 'count_2021'
})

data_selected['change'] = data_selected['count_2021'] - data_selected['count_2006']
changes = data_selected.sort_values('change', ascending=False).reset_index(drop=True)
changes['rank'] = changes.index

changes_long = changes.melt(
    id_vars=['country', 'region', 'change', 'rank'],
    value_vars=['count_2006', 'count_2021'],
    var_name='year',
    value_name='visa_free_count'
)
changes_long['year'] = changes_long['year'].map({'count_2006': '2006', 'count_2021': '2021'})

start_param = alt.param('start', value=0, bind=alt.binding_range(min=0, max=len(changes)-10, step=1, name='Start Rank:'))
end_param = alt.param('end', value=10, bind=alt.binding_range(min=10, max=len(changes), step=1, name='End Rank:'))

base = alt.Chart(changes_long).encode(
    y=alt.Y('country:N',
            title='Country',
            sort=alt.EncodingSortField(field='change', order='descending'),
            axis=alt.Axis(labelLimit=200)),
    x=alt.X('change:Q', title='Visa-Free Destinations'),
    color=continent_color,
    tooltip=['country', 'region', 'year', 'visa_free_count', 'change']
).transform_filter(
    alt.datum.year == '2021'
).transform_filter(
    f"datum.rank >= start && datum.rank < end"  # ✅ correct JS signal syntax
).add_params(
    start_param,
    end_param
)

bars = base.mark_bar()
text_labels = base.mark_text(align='left', dx=3).encode(text=alt.Text('change:Q', format='+d'))

chart = (bars + text_labels).properties(
    title='',
    width=400,
    height=400
)

slope_chart = alt.Chart(changes_long).encode(
    y=alt.Y('country:N',
            sort=alt.EncodingSortField(field='change', order='descending'),
            axis=None),
    x=alt.X('visa_free_count:Q', title='Visa-Free Destinations'),
    color=continent_color,
    detail='country:N',
    tooltip=['country', 'region', 'year', 'visa_free_count', 'change']
).transform_filter(
    f"datum.rank >= start && datum.rank < end"  # ✅ same fix here
).add_params(
    start_param,
    end_param
).properties(
    width=300,
    height=400
)

lines = slope_chart.mark_line(point=True, size=3)

q2 = alt.hconcat(chart, lines).properties(
    title=alt.TitleParams(
        text="Countries with Greatest Changes in Visa-Free Access (2006–2021)",
        anchor="middle"
    )
)
q2


alt.HConcatChart(...)

# Q3: What was the impact of COVID-19 on visa-free mobility?

## Understanding the question
To know the effect covid had in the mobility, we require temporal information that is not provided in the original dataset. country_lists dataset has the latest visa information while rank_by_year does not provide the arrival information of each country, only the exits.

I was not eable to find the infromation to complete the dataset with arrivals on different years in the official page https://www.henleyglobal.com/passport-index/ranking so I just analyzed the destinies of each country.


## My Approach
For this chart, I visualized visa-free mobility over time by plotting country-level lines (transparent) and continent-level mean lines (bold) using Altair. I was expecting to see a shared pattern (lots of lines in descending direction) on the covid pandemic but it is not the case even though it shows that, in those years, it did not grow in the same speed as before. Parallel charts are useful to see this patterns and do not have that much visual noise like point charts connected with lines. To improve intuitivity over what we are seeing, I added the mean of each continent to see the overall flow.

The colors are the same as the rest of charts.




In [106]:
import altair as alt
import pandas as pd

df = country_lists.copy()

# Melt visa_free_count columns for all years
visa_free_count = df.melt(
    id_vars=['country', 'region'],
    value_vars=[col for col in df.columns if col.startswith('year_visa_free_count_')],
    var_name='year',
    value_name='visa_free_count'
)
visa_free_count['year'] = visa_free_count['year'].str.replace('year_visa_free_count_', '').astype(int)

# remove unwanted years
visa_free_count = visa_free_count[~visa_free_count['year'].isin([2007, 2009])]

all_years = sorted(visa_free_count['year'].unique())

# Mean per region(continent lines)
mean_per_region = visa_free_count.groupby(['region', 'year'])['visa_free_count'].mean().reset_index()

base = alt.Chart().encode(
    x=alt.X('year:O', title='Year', axis=alt.Axis(values=all_years))
)

parallel_chart = alt.Chart(visa_free_count).mark_line(opacity=0.15).encode(
    x=alt.X('year:Q', title='Year', axis=alt.Axis(values=all_years)),
    y=alt.Y('visa_free_count:Q', title='Visa-Free Count'),
    color=continent_color,
    detail='country:N',
    tooltip=['country', 'region', 'year', 'visa_free_count']
).properties(
    width=700,
    height=400
)

continent_chart = alt.Chart(mean_per_region).mark_line(point=True, size=3, opacity=1).encode(
    x=alt.X('year:Q', title='Year', axis=alt.Axis(values=all_years)),
    y=alt.Y('visa_free_count:Q', title='Mean Visa-Free Count'),
    color=continent_color,
    tooltip=['region', 'year', 'visa_free_count']
).properties(
    width=700,
    height=400
)

q3 = alt.layer(continent_chart, parallel_chart, ).resolve_scale(
    x='shared',
    y='shared'
).properties(
    title='Visa-Free Destinations Over Time: Countries (faint) and Continents (bold)',
    width=500,
    height=400
).interactive()

q3

alt.LayerChart(...)

# Q4: Do countries that belong to certain global alliances, or have stronger economies, tend to enjoy greater visa-free access to the US?


```Question 4 is the one that relates to the USA, so you will need to focus only on how is the visa-free policy to enter in the USA.```

We have the following ways to enter US
-   visa_online, 
-   visa_on_arrival, 
-   visa_free_access
-   electronic_travel_authorisation 
-   visa_required


To answer whether global alliances or economic strength relate to US visa-free access, I designed a geospatial map. 

Each country is colored by its US access type (visa required, electronic travel authorisation, visa online, visa on arrival, visa-free access), making it easy to visually group countries by policy. The map uses a distinct color palette for each access type, improving legibility and allowing users to quickly distinguish between levels of restriction. The United States is highlighted in red for context, and Schengen Area countries are outlined with a green stroke to emphasize alliance membership. This design helps users spot regional patterns, such as most European countries having electronic travel authorisation, and neighbors like Canada enjoying visa-free access. Tooltips provide detailed information for each country. I chose a map over bar or table formats because spatial relationships and clusters are crucial for this question. Attempts with bar charts or tables obscured geographic trends and alliances. The layered approach, clear legend, and visual emphasis on alliances make it straightforward for users to answer the question by observing color and stroke patterns on the map.

We loose the ability to visualize small countries like andorra or vatican city in exchange for a better intuitive understanding of relations of countries. It also allows to check strong economical countries and they're relation with US.

I tried to create a 7 color palete that works even on color blindness. The complexity of the color scheme I created is that we require 5 continous values to represent freedom level and three complitelly unique colors for "US", "unknown free level" and "Schenger group". I ended up using 5 blue gradient colors and three really different colors for the discrete values.


In [93]:
from vega_datasets import data

# Create a geospatial map showing visa-free count per country using Altair

# Load world topojson from Vega datasets
world_map = alt.topo_feature(data.world_110m.url, 'countries')

# Prepare country data for 2021
country_counts = country_lists.copy()

# This definition turned out to be quite useful to improve readability and check for color blindness
color=alt.Color(
    'us_access_type:N',
    title='US Access Type',
    scale=alt.Scale(
        domain=[
            'visa_required',
            'electronic_travel_authorisation',
            'visa_online',
            'visa_on_arrival',
            'visa_free_access',
            'unknown',
            'US',
            'Schengen Area'
        ],
        # First five: blue scale, last three: unique contrasting colors
        range=[
            "#c6dbef",      # visa_required (light blue)
            "#6baed6",      # electronic_travel_authorisation (medium blue)
            "#3182bd",      # visa_online (blue)
            "#08519c",      # visa_on_arrival (dark blue)
            "#08306b",      # visa_free_access (deep blue)
            "#fdae61",      # unknown (orange)
            "#e87c7e",      # US (red)
            "#31a354",      # Schengen Area (green)
        ]
    )
)

# Altair geoshape map
geo_chart = alt.Chart(world_map).mark_geoshape(
    # stroke='black',
    # strokeWidth=0.5
).transform_lookup(
    lookup='id',
    # Transform country_counts to have 'iso_numeric' and 'us_access_type_code'. Is not a required line but i prefer to be explicit
    from_=alt.LookupData(data=country_counts, key='iso_numeric', fields = ['iso_numeric', 'us_access_type_code', 'us_access_type', 'country'])
).encode(
    tooltip=['id:N', 'country:N', 'us_access_type:N'],
    # color=alt.Color('us_access_type:N', title='US Access Type', 
    #                 scale=alt.Scale(domain=['visa_required', 'electronic_travel_authorisation', 'visa_online', 'visa_on_arrival', 'visa_free_access'],
    #                                 range=["#96512c", '#fc8d59', '#fee090', '#91bfdb', '#4575b4']))
    color = color
).project(
    type='naturalEarth1'
).properties(
    title='US Access Type per Country (2021)',
    width=700,
    height=500
)

# second map layer that highlights only the US in red
us_highlight = alt.Chart(world_map).mark_geoshape(
    fill=None,
    stroke='black',
    strokeWidth=0.5
).transform_filter(
    alt.datum.id == 840  # ISO numeric code for United States
).project(
    type='naturalEarth1'
)


# List of Schengen Area ISO numeric codes (example, check for completeness)
schengen_iso_codes = [
    40, 56, 100, 191, 203, 208, 233, 246, 250, 276, 300, 348, 352, 372, 380, 428, 440, 442, 528, 578, 616, 620, 703, 705, 724, 752, 756, 807
]

schengen_highlight = alt.Chart(world_map).mark_geoshape(
    fill=None,
    stroke="#2d6205",
    strokeWidth=1,
    opacity=0.5
).transform_filter(
    alt.FieldOneOfPredicate(field='id', oneOf=schengen_iso_codes)
).project(
    type='naturalEarth1'
)


# Combine with your base map
q4 = (geo_chart  + schengen_highlight + us_highlight)
q4


alt.LayerChart(...)

# Final composition

I combined in two rows, five charts. In an "L" shape I placed the ones that share the color scheme. This helps users associate elements that are related. 

In some plots, such as the strip chart, missing data is not plotted. This affects to the mean value but it is barelly noticiable since really few countries do not have data at some year.


The final composition presents a few problems. 
* Color: we are given lots of variables and it is really hard to use complitelly different colors to make the viewer understand they are independent graphs. The two different color schemes that I used overlap in the blues and green.

* Relation: The plots do not share many qualities. This could be improved making more shared filtering but it will also affect on the way we can use the plots. For example, if we linked the "visa free destination by continent" and the "countries with greatest changes in visa free access", the slope chart will loose too many datapoints to be relevant.

### Streamlit
Additionally, in streamlit we can share actions like highlight between charts. I added selection between Q1, Q2, Q3 but for it to work, I had to place them in the same row. The plot will not be really well composed but it will offer some interesting functionalityes in exchange.



## Conclusion
It endend up being a prety simple selection of charts:
- Geospatial
- Line + Parallel Coordinates
- Bar + slope
- Strip + helper lines

They dont look related at first sight which could be improved a little bit by some kind of global interaction that is refelcted on all charts at the same time. Interaction is also limited with only three slides and a menu. We can improve this by adding click or drag interactions on the charts.



In [109]:
# Hide legend for q3 by setting legend=None in the color encoding
q3_no_legend = q3.encode(color=alt.Color('region:N', legend=None))

# Combine the three Altair charts into a single vertical layout
combined_chart = alt.vconcat(
    alt.hconcat(q2, q1),
    alt.hconcat(q3_no_legend, q4).resolve_scale(color="independent"),
).resolve_scale(color='independent')
combined_chart

alt.VConcatChart(...)